# Actividad 3.3: Limpieza de Datos

**Dataset:** Titanic (Kaggle)
**Objetivos:**
- Visualizar el estado previo de los datos
- Estandarizar nombres de columnas
- Eliminar registros duplicados
- Manejar valores nulos (imputación y eliminación)
- Convertir tipos de datos y crear nuevas características
- Mostrar el estado de los datos limpios

In [1]:
import pandas as pd
import numpy as np

# Load the dataset to ensure we have a fresh start
df = pd.read_csv("titanic.csv")

# 1. State BEFORE cleaning
print("--- State BEFORE cleaning ---")
print(f"Shape: {df.shape}")
print("\nNull values:")
print(df.isnull().sum())

--- State BEFORE cleaning ---
Shape: (891, 12)

Null values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


### 1. Estandarizar nombres de columnas
Para evitar problemas con mayúsculas y espacios invisibles, convertiremos todos los nombres de las columnas a minúsculas y eliminaremos los espacios en blanco al principio o al final.

In [2]:
# Standardize column names (lowercase and remove trailing/leading spaces)
df.columns = df.columns.str.lower().str.strip()

print("Standardized columns:")
print(df.columns.tolist())

Standardized columns:
['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


### 2. Eliminar duplicados
Revisaremos si existen filas completamente idénticas y, de ser así, las eliminaremos para no sesgar nuestros futuros análisis.

In [3]:
# Check for duplicates
duplicates_count = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicates_count}")

# Remove duplicates
df.drop_duplicates(inplace=True)

Number of duplicate rows found: 0


### 3. Manejo de valores nulos
En el dataset del Titanic, las columnas `age`, `cabin` y `embarked` suelen tener nulos.
- **Age**: Imputaremos los valores faltantes usando la mediana.
- **Embarked**: Imputaremos usando la moda (el valor más frecuente).
- **Cabin**: Como tiene demasiados valores nulos, eliminaremos la columna por completo.

In [4]:
# Impute missing values for 'age' using the median
median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age)

# Impute missing values for 'embarked' using the mode
mode_embarked = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(mode_embarked)

# Drop the 'cabin' column due to high percentage of missing values
if "cabin" in df.columns:
    df.drop("cabin", axis=1, inplace=True)

### 4. Crear nuevas columnas y convertir tipos de datos
Crearemos una nueva columna llamada `family_size` sumando las columnas `sibsp` (hermanos/esposo) y `parch` (padres/hijos) más 1 (el propio pasajero). También crearemos una columna booleana `is_alone`. Finalmente, ajustaremos los tipos de datos.

In [5]:
# Create new columns
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["is_alone"] = (df["family_size"] == 1).astype(int)

# Convert data types
# Age is better represented as an integer
df["age"] = df["age"].astype(int)

# Survived, pclass, and is_alone are categorical variables disguised as numeric
df["survived"] = df["survived"].astype("category")
df["pclass"] = df["pclass"].astype("category")
df["is_alone"] = df["is_alone"].astype("category")

### 5. Estado de los datos DESPUÉS de la limpieza
Verificamos que nuestra limpieza se haya aplicado correctamente, confirmando el nuevo tamaño del dataframe, la ausencia de nulos y los nuevos tipos de datos.

In [6]:
# State AFTER cleaning
print("--- State AFTER cleaning ---")
print(f"New Shape: {df.shape}\n")

print("--- Null values ---")
print(df.isnull().sum())

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- First 5 rows of clean dataset ---")
df.head()

--- State AFTER cleaning ---
New Shape: (891, 13)

--- Null values ---
passengerid    0
survived       0
pclass         0
name           0
sex            0
age            0
sibsp          0
parch          0
ticket         0
fare           0
embarked       0
family_size    0
is_alone       0
dtype: int64

--- Data Types ---
passengerid       int64
survived       category
pclass         category
name                str
sex                 str
age               int64
sibsp             int64
parch             int64
ticket              str
fare            float64
embarked            str
family_size       int64
is_alone       category
dtype: object

--- First 5 rows of clean dataset ---


,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,embarked,family_size,is_alone
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.2500,S,2,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,71.2833,C,2,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.9250,S,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1000,S,2,0
4,5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.0500,S,1,1
